# PRÁCTICA: Autoencoder Estándar con MNIST

**Nombre:** ___________________________  
**Fecha:** ___________________________

---

## Objetivo

En esta práctica construirás y entrenarás un **autoencoder estándar** con capas densas sobre el dataset **MNIST** (dígitos escritos a mano del 0 al 9).  
Al finalizar serás capaz de:

- Definir la arquitectura de un autoencoder (encoder + decoder).
- Entrenar el modelo para que aprenda a reconstruir imágenes.
- Visualizar las reconstrucciones y el espacio latente.
- Experimentar con distintos tamaños del espacio latente y analizar el impacto.

---

## Descripción del dataset

MNIST contiene **70.000 imágenes** en escala de grises de dígitos escritos a mano (0–9), de 28×28 píxeles.

```
Forma de los datos:
  x_train → (60000, 28, 28)
  x_test  → (10000, 28, 28)
```

Para alimentar un autoencoder denso, aplanaremos cada imagen a un vector de **784 valores**.


## Paso 1 – Importar librerías

Ejecuta la celda sin modificaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import mnist

print("TensorFlow version:", tf.__version__)

## Paso 2 – Cargar y preprocesar los datos

**Tarea:** completa las dos líneas marcadas con `# TODO` para:
1. Normalizar los píxeles al rango [0, 1].
2. Aplanar cada imagen de (28, 28) a (784,).

> **Pista:** usa `.astype('float32') / 255.` para normalizar, y `.reshape(-1, 784)` para aplanar.

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# TODO: normaliza x_train y x_test al rango [0,1]
x_train = ???
x_test  = ???

# TODO: aplana cada imagen a un vector de 784 elementos
x_train_flat = ???
x_test_flat  = ???

print("Forma train aplanado:", x_train_flat.shape)
print("Forma test aplanado: ", x_test_flat.shape)

## Paso 3 – Visualizar algunas imágenes de ejemplo

Ejecuta sin modificaciones para familiarizarte con los datos.

In [ ]:
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title(str(y_test[i]))
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.suptitle("Imágenes originales del conjunto de test")
plt.show()

## Paso 4 – Definir la arquitectura del Autoencoder

Vas a construir un autoencoder con la siguiente arquitectura simétrica:

```
Entrada  →  784
  Encoder:
    Dense(128, relu)
    Dense(64,  relu)
    Dense(LATENT_DIM, relu)   ← espacio latente (cuello de botella)
  Decoder:
    Dense(64,  relu)
    Dense(128, relu)
    Dense(784, sigmoid)       ← reconstrucción
Salida   →  784
```

**Tarea:** completa las líneas marcadas con `# TODO`.

In [ ]:
LATENT_DIM = 32   # tamaño del espacio latente (puedes cambiarlo más adelante)

class Autoencoder(Model):
    def __init__(self, latent_dim):
        super(Autoencoder, self).__init__()
        
        # --- ENCODER ---
        self.encoder = tf.keras.Sequential([
            layers.Input(shape=(784,)),
            # TODO: añade Dense(128) con activación 'relu'
            ???,
            # TODO: añade Dense(64) con activación 'relu'
            ???,
            # TODO: añade la capa del espacio latente Dense(latent_dim) con activación 'relu'
            ???
        ])
        
        # --- DECODER ---
        self.decoder = tf.keras.Sequential([
            layers.Input(shape=(latent_dim,)),
            # TODO: añade Dense(64) con activación 'relu'
            ???,
            # TODO: añade Dense(128) con activación 'relu'
            ???,
            # TODO: añade la capa de salida Dense(784) con activación 'sigmoid'
            ???
        ])

    def call(self, x):
        encoded  = self.encoder(x)
        decoded  = self.decoder(encoded)
        return decoded


autoencoder = Autoencoder(LATENT_DIM)

# TODO: compila el modelo con optimizador 'adam' y función de pérdida 'mse'
autoencoder.compile(optimizer=???, loss=???)

autoencoder.encoder.summary()
autoencoder.decoder.summary()

## Paso 5 – Entrenar el modelo

**Tarea:** completa la llamada a `.fit()` con los parámetros correctos.  

> **Recuerda:** en un autoencoder, la entrada **y** la salida son los **mismos datos**.

In [ ]:
history = autoencoder.fit(
    x_train_flat,       # datos de entrada
    ???,                # TODO: ¿cuál es el target (salida esperada)?
    epochs=???,         # TODO: usa 20 épocas
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, ???)  # TODO: completa el target de validación
)

## Paso 6 – Visualizar la curva de pérdida

Ejecuta sin modificaciones.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Evolución de la pérdida')
plt.legend()
plt.grid(True)
plt.show()

## Paso 7 – Reconstruir imágenes y visualizar resultados

**Tarea:** completa la línea de predicción.

> **Pista:** usa `autoencoder.predict(...)` con los datos de test aplanados.

In [ ]:
# TODO: obtén las reconstrucciones del conjunto de test
reconstructions = ???

n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    # Imagen original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i], cmap='gray')
    plt.title("original")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Imagen reconstruida
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructions[i].reshape(28, 28), cmap='gray')
    plt.title("reconstruido")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.suptitle(f'Reconstrucciones del Autoencoder (espacio latente = {LATENT_DIM})')
plt.show()

## Paso 8 – Visualizar el espacio latente (encoder)

Proyectaremos las representaciones latentes en 2D con PCA para ver si el encoder ha aprendido a separar los dígitos.

Ejecuta sin modificaciones.

In [ ]:
from sklearn.decomposition import PCA

# Obtenemos el espacio latente de los datos de test
encoded_test = autoencoder.encoder.predict(x_test_flat)

# Reducimos a 2D con PCA para poder visualizar
pca = PCA(n_components=2)
encoded_2d = pca.fit_transform(encoded_test)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(encoded_2d[:, 0], encoded_2d[:, 1],
                      c=y_test, cmap='tab10', alpha=0.5, s=5)
plt.colorbar(scatter, label='Dígito')
plt.xlabel('Componente PCA 1')
plt.ylabel('Componente PCA 2')
plt.title(f'Espacio latente proyectado en 2D (latent_dim={LATENT_DIM})')
plt.grid(True)
plt.show()

## Paso 9 – Experimentación: efecto del tamaño del espacio latente

**Tarea:** repite el entrenamiento con **dos valores distintos** de `LATENT_DIM` (por ejemplo, 2 y 128) y responde a las preguntas de reflexión.

Puedes copiar las celdas anteriores o simplemente cambiar `LATENT_DIM` al principio y volver a ejecutar todo.

In [ ]:
# Prueba con LATENT_DIM = 2
# Prueba con LATENT_DIM = 128
# Anota la pérdida final de validación en cada caso y observa la calidad de las reconstrucciones

resultados = {
    'LATENT_DIM': [],
    'val_loss_final': []
}

for dim in [2, 32, 128]:
    ae = Autoencoder(dim)
    ae.compile(optimizer='adam', loss='mse')
    hist = ae.fit(x_train_flat, x_train_flat,
                  epochs=20, batch_size=256,
                  validation_data=(x_test_flat, x_test_flat),
                  verbose=0)
    val_loss = hist.history['val_loss'][-1]
    resultados['LATENT_DIM'].append(dim)
    resultados['val_loss_final'].append(round(val_loss, 5))
    print(f"LATENT_DIM={dim:4d}  →  val_loss={val_loss:.5f}")

print("\nTabla resumen:", resultados)

## Preguntas de reflexión

Responde en las celdas de texto a continuación.

---

**P1.** ¿Qué ocurre con la calidad de las reconstrucciones cuando el espacio latente es muy pequeño (por ejemplo, 2 dimensiones)?  
¿Y cuando es muy grande (128)? ¿Por qué?

**Respuesta P1:**  

_(escribe aquí)_

**P2.** Observa el gráfico del espacio latente proyectado en 2D. ¿Consigue el encoder separar los dígitos en regiones distintas? ¿Qué dígitos parecen más difíciles de separar y por qué crees que es así?

**Respuesta P2:**  

_(escribe aquí)_

**P3.** En el entrenamiento, el target (salida esperada) es idéntico a la entrada. ¿Por qué tiene sentido esto en un autoencoder? ¿Qué aprende realmente la red?

**Respuesta P3:**  

_(escribe aquí)_

**P4 (opcional / avanzada).** El autoencoder que has construido usa únicamente capas densas (`Dense`). ¿Qué ventajas podría tener usar capas convolucionales (`Conv2D`) en lugar de capas densas para trabajar con imágenes?

**Respuesta P4:**  

_(escribe aquí)_

---

## Entrega

Sube este notebook con:
- Todos los `???` completados.
- Las celdas ejecutadas (con output visible).
- Las respuestas de reflexión escritas.

**Criterios de evaluación:**

| Criterio | Puntos |
|---|---|
| Preprocesamiento correcto (Pasos 2) | 1 |
| Arquitectura del autoencoder correcta (Paso 4) | 3 |
| Entrenamiento correcto — target bien definido (Paso 5) | 2 |
| Reconstrucciones visualizadas y coherentes (Paso 7) | 1 |
| Experimentación con distintos LATENT_DIM (Paso 9) | 1 |
| Preguntas de reflexión P1–P3 respondidas | 2 |
| **Total** | **10** |
